In [71]:

import numpy as np
from numpy.typing import NDArray
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.axes import Axes
import astropy.units as u
from astropy.io import fits
from astropy.wcs import WCS
from astropy.time import Time
from astropy.nddata import Cutout2D
from astropy.stats import sigma_clipped_stats
from astropy.coordinates import SkyCoord, EarthLocation, Angle, get_body_barycentric
from photutils.segmentation import detect_sources, SegmentationImage, SourceCatalog
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares, OptimizeResult
import os
BOX_SIZE = 200

In [72]:
os.chdir("/home/odin/Code/Course-Assignments/PH556_Astrophysics/Attempt01/data/")
print("Current working directory: {0}".format(os.getcwd()))
allDataFileNames = os.listdir()
# print("Files in current working directory: {0}".format(allDataFileNames))
print(allDataFileNames)

files = [fits.open(filename) for filename in allDataFileNames]
# fileData = [file[0].data for file in files]
# fileHeaders = [file[0].header for file in files]

Current working directory: /home/odin/Code/Course-Assignments/PH556_Astrophysics/Attempt01/data
['20210721192429-359-RA.wcs.proc.fits', '20210721191920-421-RA.wcs.proc.fits', '20210713215458-181-RA.wcs.proc.fits', '20210721193447-216-RA.wcs.proc.fits', '20210721191411-345-RA.wcs.proc.fits', '20210713220516-046-RA.wcs.proc.fits', '20210721194505-080-RA.wcs.proc.fits', '20210713213912-742-RA.wcs.proc.fits', '20210713214440-179-RA.wcs.proc.fits', '20210713214949-265-RA.wcs.proc.fits', '20210713221024-980-RA.wcs.proc.fits', '20210714213919-746-RA.wcs.proc.fits', '20210713221533-914-RA.wcs.proc.fits', '20210721195014-013-RA.wcs.proc.fits', '20210714220020-444-RA.wcs.proc.fits']


In [73]:
# '2021-07-21T19:24:29.359'

def parseOBSDate(dateString):
    res1 = dateString.split("T")
    resMajor = res1[0].split("-")
    resMinor = res1[1].split(":")
    res = [float(x) for x in resMajor + resMinor]
    numVal = res[0]*(12*31)*24*60*60 + res[1]*31*24*60*60 + res[2]*24*60*60 + res[3]*60*60 + res[4]*60 + res[5]
    return [res, numVal]

def addTime(res, numVal, dtInSeconds):
    numValNew = numVal + dtInSeconds
    resNew = [0, 0, 0, 0, 0, 0]
    resNew[0] = res[0]
    resNew[1] = res[1]
    resNew[2] = res[2]
    resNew[3] = res[3]
    resNew[4] = res[4]
    resNew[5] = res[5] + dtInSeconds
    if resNew[5] >= 60:
        resNew[4] += int(resNew[5]/60)
        resNew[5] = resNew[5]%60
    if resNew[4] >= 60:
        resNew[3] += int(resNew[4]/60)
        resNew[4] = resNew[4]%60
    if resNew[3] >= 24:
        resNew[2] += int(resNew[3]/24)
        resNew[3] = resNew[3]%24
    if resNew[2] > 31:
        resNew[1] += int(resNew[2]/31)
        resNew[2] = resNew[2]%31
    if resNew[1] > 12:
        resNew[0] += int(resNew[1]/12)
        resNew[1] = resNew[1]%12
    return [resNew, numValNew]

In [74]:
datesOfObservationParsed = [parseOBSDate(file[0].header['DATE-OBS']) for file in files]
datesOfObservationNumVal = [date[1] for date in datesOfObservationParsed]
datesOfObservationDateVal = [date[0] for date in datesOfObservationParsed]
indicesForDateSortingAscending = np.argsort(datesOfObservationNumVal)

In [75]:
filesAscendingDate = [files[i] for i in indicesForDateSortingAscending]
dataValsArranged = [file[0].data for file in filesAscendingDate]
headerValsArranged = [file[0].header for file in filesAscendingDate]

In [76]:
(Time(files[1][0].header['DATE-OBS'],scale='utc') + files[1][0].header['EXPTIME']*u.s/2).jd

np.float64(2459417.3068335764)

In [77]:
bin1Files = filesAscendingDate[0:7]
bin2Files = filesAscendingDate[7:9]
bin3Files = filesAscendingDate[9:15]
bins = [bin1Files, bin2Files, bin3Files]
binRanges = [[i for i in range(0,7)], [i for i in range(7,9)], [i for i in range(9,15)]]
binRanges

[[0, 1, 2, 3, 4, 5, 6], [7, 8], [9, 10, 11, 12, 13, 14]]

In [78]:
def getMidExposureTime(file):
    dateObs = file[0].header['DATE-OBS']
    exptime = file[0].header['EXPTIME']
    midExposureTime = (Time(dateObs, scale='utc') + exptime*u.s/2).jd
    return midExposureTime

midExposureTimes = [getMidExposureTime(filesAscendingDate[i]) for i in range(len(filesAscendingDate))]

In [79]:
RAVals = [headerValsArranged[i]['TARRA'] for i in range(len(headerValsArranged))]
DecVals = [headerValsArranged[i]['TARDEC'] for i in range(len(headerValsArranged))]


In [68]:
def subsetTakeAndFilter(file, centerRA, centerDec, size = BOX_SIZE):
    data = file[0].data
    header = file[0].header
    wcs = WCS(header)
    subset = Cutout2D(data, position=SkyCoord(centerRA, centerDec, unit=(u.deg, u.deg)), size=(size, size), wcs=wcs)
    _, sky, sigma = sigma_clipped_stats(subset.data)
    return subset.data - sky, subset.wcs, float(sigma)


def find_sources(img, sigma: float, nsigma: float = 3.0,
                 npix: int = 8):
    '''Pixel (x, y) centres of all blobs with >= npix connected pixels above nsigma*sigma.'''
    seg: SegmentationImage | None = detect_sources(img, nsigma * sigma, npix)
    if seg is None:                                           # nothing above threshold
        return np.array([]), np.array([])
    cat: SourceCatalog = SourceCatalog(img, seg)
    return np.asarray(cat.xcentroid, float), np.asarray(cat.ycentroid, float)


def predicted_position(hdr0: fits.Header, t0: Time, t: Time) -> SkyCoord:
    dt_hr: float = (t - t0).to_value(u.hour)
    ra: float = hdr0['TARRA'] + float(hdr0['DRATERA']) * dt_hr    
    dec: float = hdr0['TARDEC'] + float(hdr0['DRATEDEC']) * dt_hr
    return SkyCoord(ra, dec, unit='deg')


In [69]:
CENTROID_R: float = 6.0
def centroid(img, x: float, y: float, r: float = CENTROID_R,
             n_iter: int = 5) -> tuple[float, float]:
    '''Iterative flux-weighted centroid inside a circle of radius r around (x, y). Pixel units.'''
    yy: NDArray[np.intp]
    xx: NDArray[np.intp]
    yy, xx = np.indices(img.shape)                  # pixel coordinate grids, each (ny, nx)
    for _ in range(n_iter):
        inside = (xx - x)**2 + (yy - y)**2 <= r**2     # (ny, nx) mask
        w: Image = np.clip(img * inside, 0, None)   # negative (noise) pixels get zero weight
        x, y = float((w * xx).sum() / w.sum()), float((w * yy).sum() / w.sum())
    return x, y

In [ ]:
subsetImageData = []
subsetImageWCS = []
subsetImagesSources = []

STAR_MOVEMENT_THRESHOLD = 2.0*u.arcsec

ASTEROID_SEARCH_RADIUS = 5


for nightIdx in range(len(binRanges)):
    for fileIdx in binRanges[nightIdx]:
        imgData, fileWCS, sigma =  subsetTakeAndFilter(filesAscendingDate[fileIdx], RAVals[fileIdx], DecVals[fileIdx], size=BOX_SIZE)
        xs, ys = find_sources(imgData, sigma)
        subsetImageData.append(imgData)
        subsetImageWCS.append(fileWCS)
        skyWCS = fileWCS.pixel_to_world(xs, ys)
        subsetImagesSources.append(skyWCS)

        currentMidExposureTime = midExposureTimes[fileIdx]
        indexOfFarthestExposureTime = 0
        if (currentMidExposureTime - midExposureTimes[binRanges[nightIdx][0]]) > (midExposureTimes[binRanges[nightIdx][-1]] - currentMidExposureTime):
            indexOfFarthestExposureTime = binRanges[-1][-1]
        _, d_far, _ = skyWCS.match_to_catalog(subsetImageWCS[indexOfFarthestExposureTime])
        isPixelMoving = d_far > STAR_MOVEMENT_THRESHOLD

        predictedAsteroidPos = predicted_position(headerValsArranged[0], Time(midExposureTimes[0], format='jd'), Time(currentMidExposureTime, format='jd'))
        nearbyPixelDistances = skyWCS.separation(predictedAsteroidPos).arcsec
        nearbyPixelDistances[~isPixelMoving] = np.inf
        closestSourceIdx: int | None = int(np.argmin(nearbyPixelDistances)) if len(nearbyPixelDistances) > 0 else None
        asteroidDetected = closestSourceIdx is not None and nearbyPixelDistances[closestSourceIdx] < ASTEROID_SEARCH_RADIUS